# Modelo predictivo final — Criminalidad en Chicago

Modelo predictivo desarrollado conjuntamente para estimar el número de crímenes por
Community Area en 2025-2027, a partir de variables socioeconómicas anualizadas.

**Contenido:**
1. Preparación de datos: crímenes agregados por zona y año + variables socioeconómicas
2. Modelo Random Forest (modelo final) y evaluación frente a un modelo base (Dummy)
3. Modelos de comparación: Regresión Lineal y OLS (statsmodels)
4. Predicción a 3 años (2025-2027) con proyección de variables socioeconómicas
5. Análisis complementario: tipos de crimen y tasas por Community Area


## 1. Carga y preparación de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.api as sm

pd.set_option("display.max_columns", None)

In [2]:
# Dataset de crímenes (mismo origen que en el EDA)
df = pd.read_csv("../data/Proyecto Bootcamp.csv")

df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p")
df["Year"] = df["Date"].dt.year

df.shape

(1456714, 23)

In [3]:
# Agregamos el número de crímenes por Community Area y año
crimes_area_year = (
    df
    .groupby(["Year", "Community Area"])
    .size()
    .reset_index(name="Crime_Count")
)

# Community Area 0 es un caso atípico (un único registro mal etiquetado, ver EDA)
crimes_area_year = crimes_area_year[crimes_area_year["Community Area"] != 0]
crimes_area_year = crimes_area_year.reset_index(drop=True)

crimes_area_year.head()

,Year,Community Area,Crime_Count
0,2020,1.0,4923
1,2020,2.0,4028
2,2020,3.0,4976
3,2020,4.0,2339
4,2020,5.0,1920


### Variables socioeconómicas por Community Area y año

Fuente: variables anualizadas de población, renta per cápita, tasa de pobreza, tasa de
desempleo y nivel educativo (Educación secundaria) por Community Area, 2020-2024.

In [5]:
variables = pd.read_excel("../data/Variables_socioeconomicas.xlsx")
variables.columns

Index(['Layer', 'Name', 'GEOID', 'Year', 'Longitude', 'Latitude', 'Population',
       'Per Capita Income', 'Poverty Rate', 'Unemployment Rate',
       'Education Rate (High School)'],
      dtype='object')

In [6]:
crimes_area_year["Community Area"] = crimes_area_year["Community Area"].astype(int)
crimes_area_year["Year"] = crimes_area_year["Year"].astype(int)

variables["GEOID"] = variables["GEOID"].astype(int)
variables["Year"] = variables["Year"].astype(int)

df_model = crimes_area_year.merge(
    variables,
    left_on=["Community Area", "Year"],
    right_on=["GEOID", "Year"],
    how="left"
)

df_model.head()

,Year,Community Area,Crime_Count,Layer,Name,GEOID,Longitude,Latitude,Population,Per Capita Income,Poverty Rate,Unemployment Rate,Education Rate (High School)
0,2020,1,4923,Community area,Rogers Park,1.0,-87.670171,42.009630,54290.419673,37643.137976,16.749840,11.589611,87.894224
1,2020,2,4028,Community area,West Ridge,2.0,-87.695017,42.001583,78777.599098,32621.185986,18.602764,12.260817,86.559054
2,2020,3,4976,Community area,Uptown,3.0,-87.655899,41.965822,54760.207860,51866.536153,17.860233,7.945636,91.823313
3,2020,4,2339,Community area,Lincoln Square,4.0,-87.687513,41.975182,41770.467212,58998.701919,9.097879,8.738618,94.229999
4,2020,5,1920,Community area,North Center,5.0,-87.683839,41.947811,36225.402334,83620.270350,4.836840,6.424326,96.756573


In [7]:
numeric_cols = df_model.select_dtypes(include=["int64", "float64"])
correlation_crime = numeric_cols.corr()["Crime_Count"].sort_values(ascending=False)
correlation_crime

Crime_Count                     1.000000
Population                      0.624157
Unemployment Rate               0.247804
Poverty Rate                    0.212100
Longitude                       0.044065
Latitude                        0.026755
Per Capita Income               0.017644
Education Rate (High School)   -0.058161
Community Area                 -0.067711
GEOID                          -0.080638
Year                           -0.344276
Name: Crime_Count, dtype: float64

**Población** es, con diferencia, la variable más correlacionada con el número de
crímenes (0.62), seguida de la **tasa de desempleo** (0.25) y la **tasa de pobreza** (0.21).

## 2. Modelo final: Random Forest

In [8]:
features = [
    "Year",
    "Population",
    "Unemployment Rate",
    "Poverty Rate",
    "Per Capita Income",
    "Education Rate (High School)"
]
target = "Crime_Count"

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    max_depth=None
)
model.fit(X_train, y_train)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE: ", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:  ", round(r2, 4))

MAE:  331.33
RMSE: 541.49
R2:   0.9774


**Resultado: R² ≈ 0.977** — el modelo explica el 97.7% de la variabilidad del número de crímenes por Community Area y año.

### Comparación frente a un modelo base (Dummy)

Para confirmar que el resultado del Random Forest es realmente bueno (y no un artefacto de los datos), se compara contra un modelo *dummy* que siempre predice la media.

In [10]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
y_dummy_pred = dummy.predict(X_test)

mae_dummy = mean_absolute_error(y_test, y_dummy_pred)
rmse_dummy = np.sqrt(mean_squared_error(y_test, y_dummy_pred))
r2_dummy = r2_score(y_test, y_dummy_pred)

evaluation = pd.DataFrame({
    "Model": ["Random Forest", "Dummy (media)"],
    "MAE": [mae, mae_dummy],
    "RMSE": [rmse, rmse_dummy],
    "R2": [r2, r2_dummy]
})
evaluation

,Model,MAE,RMSE,R2
0,Random Forest,331.331365,541.487075,0.977379
1,Dummy (media),2636.724393,3601.083449,-0.000446


El Random Forest reduce el error medio (MAE) en casi un 87% frente al modelo base, y pasa de un R² prácticamente nulo (-0.0004) a 0.977.

## 3. Predicción para 2025

In [11]:
df_2025 = df_model[df_model["Year"] == 2024].copy()
df_2025["Year"] = 2025

X_2025 = df_2025[features]
df_2025["Predicted_Crime_Count_2025"] = model.predict(X_2025)
df_2025["Predicted_Crime_Count_2025"] = (
    df_2025["Predicted_Crime_Count_2025"].round().astype(int)
)

predicciones_2025 = df_2025[
    ["Community Area", "Name", "Year", "Predicted_Crime_Count_2025"] + features[1:]
].sort_values(by="Predicted_Crime_Count_2025", ascending=False)

predicciones_2025.head(10)

,Community Area,Name,Year,Predicted_Crime_Count_2025,Population,Unemployment Rate,Poverty Rate,Per Capita Income,Education Rate (High School)
332,25,Austin,2025,16693,97188.562239,12.567347,23.816378,27276.701684,83.211876
315,8,Near North Side,2025,9920,98801.927033,2.762768,9.794052,131500.747880,98.665958
335,28,Near West Side,2025,8520,67761.207974,5.760284,16.400780,93608.781283,95.736463
336,29,North Lawndale,2025,8263,33812.041678,12.458872,32.176038,26159.177302,84.359400
339,32,Loop,2025,8126,43540.667170,4.339026,11.344620,108472.083257,97.245339
350,43,South Shore,2025,8120,51123.760873,12.872862,29.058168,33243.279177,88.574949
330,23,Humboldt Park,2025,7785,55491.436156,8.757188,27.250185,30155.398528,78.490650
331,24,West Town,2025,7731,87746.010453,3.238034,8.258057,101749.040759,95.562364
378,71,Auburn Gresham,2025,7699,43644.003610,14.393086,28.509785,27610.227986,86.319795
376,69,Greater Grand Crossing,2025,7286,29074.825301,14.274999,30.322002,27932.428817,85.362094


In [12]:
total_crimenes_2025 = predicciones_2025["Predicted_Crime_Count_2025"].sum()
total_crimenes_2024 = df_model[df_model["Year"] == 2024]["Crime_Count"].sum()

print(f"Total predicho 2025: {total_crimenes_2025:,}")
print(f"Total real 2024:     {total_crimenes_2024:,}")

Total predicho 2025: 269,030
Total real 2024:     265,462


La predicción total para 2025 es muy cercana al valor real de 2024 (diferencia menor al 1.5%), lo cual es coherente con la tendencia bajista observada en el EDA.

## 4. Modelos de comparación

### Regresión Lineal

In [13]:
df_linear = df_model[features + [target]].dropna().copy()

X_lin = df_linear[features]
y_lin = df_linear[target]

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin, test_size=0.2, random_state=42
)

linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
linear_pipeline.fit(X_train_lin, y_train_lin)

y_pred_linear = linear_pipeline.predict(X_test_lin)

mae_linear = mean_absolute_error(y_test_lin, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test_lin, y_pred_linear))
r2_linear = r2_score(y_test_lin, y_pred_linear)

print("MAE: ", round(mae_linear, 2))
print("RMSE:", round(rmse_linear, 2))
print("R2:  ", round(r2_linear, 4))

MAE:  1390.69
RMSE: 1880.59
R2:   0.633


In [14]:
linear_coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": linear_pipeline.named_steps["model"].coef_
}).sort_values(by="Coefficient", ascending=False)

linear_coefficients

,Feature,Coefficient
1,Population,2710.062017
2,Unemployment Rate,1381.764367
5,Education Rate (High School),307.399210
3,Poverty Rate,298.805191
0,Year,239.398916
4,Per Capita Income,-600.659213


La Regresión Lineal alcanza un R² ≈ 0.63 — claramente inferior al Random Forest, lo que indica que las relaciones entre las variables socioeconómicas y la criminalidad no son puramente lineales.

### OLS (statsmodels) — para inferencia estadística

In [15]:
X_ols = sm.add_constant(df_linear[features])
y_ols = df_linear[target]

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:            Crime_Count   R-squared:                       0.642
Model:                            OLS   Adj. R-squared:                  0.636
Method:                 Least Squares   F-statistic:                     112.8
Date:                Thu, 25 Jun 2026   Prob (F-statistic):           4.22e-81
Time:                        13:42:06   Log-Likelihood:                -3458.5
No. Observations:                 385   AIC:                             6931.
Df Residuals:                     378   BIC:                             6959.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const           

## 5. Proyección a 3 años (2025-2027)

Para una proyección más allá de 2025, se estiman tasas de crecimiento anual de cada variable socioeconómica a partir de su evolución 2020-2024, y se aplican a los años futuros antes de predecir con el modelo Random Forest.

In [16]:
socio_vars = [
    "Population", "Unemployment Rate", "Poverty Rate",
    "Per Capita Income", "Education Rate (High School)"
]

annual_means = df_model.groupby("Year")[socio_vars].mean().reset_index()
annual_means

,Year,Population,Unemployment Rate,Poverty Rate,Per Capita Income,Education Rate (High School)
0,2020,35284.235750,16.806601,18.126414,40306.090673,85.592936
1,2021,35197.502574,10.994973,19.391047,40560.497858,85.885729
2,2022,35110.769398,7.853552,19.250533,43122.017997,86.080923
3,2023,35024.036222,7.068197,18.547958,45335.698222,86.178521
4,2024,34937.303047,8.481836,19.250533,46563.832027,86.471313
5,2025,NaN,NaN,NaN,NaN,NaN


In [17]:
growth_rates = {}
for var in socio_vars:
    value_2020 = annual_means.loc[annual_means["Year"] == 2020, var].values[0]
    value_2024 = annual_means.loc[annual_means["Year"] == 2024, var].values[0]
    growth_rates[var] = (value_2024 / value_2020) ** (1 / 4) - 1

growth_rates

{'Population': np.float64(-0.0024672441550066537),
 'Unemployment Rate': np.float64(-0.15714572186677933),
 'Poverty Rate': np.float64(0.015155832602194108),
 'Per Capita Income': np.float64(0.03673917420587003),
 'Education Rate (High School)': np.float64(0.002555749933131013)}

In [26]:
base_2024 = df_model[df_model["Year"] == 2024].copy()
future_years = [2025, 2026, 2027]
future_dfs = []

for year in future_years:
    df_future_year = base_2024.copy()
    df_future_year["Year"] = year
    years_after_2024 = year - 2024
    for var in socio_vars:
        df_future_year[var] = (
            df_future_year[var] * ((1 + growth_rates[var]) ** years_after_2024)
        )
    future_dfs.append(df_future_year)

df_future_2025_2027 = pd.concat(future_dfs, ignore_index=True)

X_future = df_future_2025_2027[features]
df_future_2025_2027["Predicted_Crime_Count"] = model.predict(X_future).round().astype(int)

predicciones_2025_2027 = df_future_2025_2027[
    ["Community Area", "Name", "Year", "Predicted_Crime_Count"] + socio_vars
].sort_values(by=["Year", "Predicted_Crime_Count"], ascending=[True, False])

predicciones_2025_2027.head(15)

,Community Area,Name,Year,Predicted_Crime_Count,Population,Unemployment Rate,Poverty Rate,Per Capita Income,Education Rate (High School)
24,25,Austin,2025,16507,96948.774327,10.592443,24.177335,28278.825179,83.424545
7,8,Near North Side,2025,9426,98558.158556,2.328610,9.942489,136331.976764,98.918123
42,43,South Shore,2025,8225,50997.626072,10.849947,29.498569,34464.609802,88.801325
28,29,North Lawndale,2025,7995,33728.619116,10.501013,32.663692,27120.243874,84.575002
70,71,Auburn Gresham,2025,7915,43536.323197,12.131274,28.941874,28624.604962,86.540407
22,23,Humboldt Park,2025,7821,55354.525235,7.381033,27.663184,31263.282967,78.691253
27,28,Near West Side,2025,7440,67594.024530,4.855080,16.649347,97047.890606,95.981142
66,67,West Englewood,2025,7300,28102.323645,12.291146,33.056865,23377.777873,77.243591
68,69,Greater Grand Crossing,2025,7264,29003.090608,12.031744,30.781557,28958.643186,85.580258
23,24,West Town,2025,7169,87529.519622,2.729190,8.383215,105487.216492,95.806598


In [19]:
totales_predichos = (
    predicciones_2025_2027
    .groupby("Year")["Predicted_Crime_Count"]
    .sum()
    .reset_index()
)
totales_predichos

,Year,Predicted_Crime_Count
0,2025,264644
1,2026,268640
2,2027,266568


In [20]:
import os
os.makedirs("../outputs", exist_ok=True)

predicciones_2025_2027.to_csv(
    "../outputs/predicciones_random_forest_2025_2027.csv",
    index=False, sep=";", decimal=","
)

## 6. Análisis complementario: tipos de crimen y tasas por zona

In [21]:
df_crimes = df[df["Community Area"] != 0].copy()

crime_type_area = (
    df_crimes
    .groupby(["Community Area", "Primary Type"])
    .size()
    .reset_index(name="Crime_Count")
)

total_area = (
    df_crimes
    .groupby("Community Area")
    .size()
    .reset_index(name="Total_Crimes_Area")
)

crime_type_area_pct = crime_type_area.merge(total_area, on="Community Area", how="left")
crime_type_area_pct["Crime_Percentage"] = (
    crime_type_area_pct["Crime_Count"] / crime_type_area_pct["Total_Crimes_Area"] * 100
).round(2)

top5_crime_types_area = (
    crime_type_area_pct
    .sort_values(by=["Community Area", "Crime_Percentage"], ascending=[True, False])
    .groupby("Community Area")
    .head(5)
    .reset_index(drop=True)
)

top5_crime_types_area.head(10)

,Community Area,Primary Type,Crime_Count,Total_Crimes_Area,Crime_Percentage
0,1.0,THEFT,4688,20500,22.87
1,1.0,BATTERY,4110,20500,20.05
2,1.0,CRIMINAL DAMAGE,2293,20500,11.19
3,1.0,NARCOTICS,1440,20500,7.02
4,1.0,OTHER OFFENSE,1414,20500,6.90
5,2.0,THEFT,4219,17736,23.79
6,2.0,BATTERY,3049,17736,17.19
7,2.0,CRIMINAL DAMAGE,2395,17736,13.50
8,2.0,BURGLARY,1326,17736,7.48
9,2.0,OTHER OFFENSE,1220,17736,6.88


In [22]:
top5_crime_types_area.to_csv(
    "../outputs/top5_crime_types_by_community_area.csv",
    index=False, sep=";", decimal=","
)

### Tasas de arresto, violencia doméstica y criminalidad violenta por zona

In [23]:
violent_crimes = [
    "HOMICIDE", "CRIMINAL SEXUAL ASSAULT", "ROBBERY", "ASSAULT",
    "BATTERY", "KIDNAPPING", "SEX OFFENSE", "OFFENSE INVOLVING CHILDREN",
    "HUMAN TRAFFICKING"
]

df_crimes_recent = df[
    (df["Year"].between(2020, 2024)) & (df["Community Area"] != 0)
].copy()
df_crimes_recent["Violent"] = df_crimes_recent["Primary Type"].isin(violent_crimes)

rates_area = (
    df_crimes_recent
    .groupby("Community Area")
    .agg(
        Total_Crimes=("ID", "count"),
        Arrest_Rate=("Arrest", "mean"),
        Domestic_Rate=("Domestic", "mean"),
        Violent_Rate=("Violent", "mean")
    )
    .reset_index()
)

for col in ["Arrest_Rate", "Domestic_Rate", "Violent_Rate"]:
    rates_area[col] = (rates_area[col] * 100).round(2)

rates_area.head()

,Community Area,Total_Crimes,Arrest_Rate,Domestic_Rate,Violent_Rate
0,1.0,20350,26.20,14.53,31.77
1,2.0,17577,17.98,13.63,28.40
2,3.0,20242,29.09,8.99,27.17
3,4.0,10181,17.81,9.29,24.72
4,5.0,7954,13.96,6.25,16.58


In [24]:
population_area = (
    df_model[(df_model["Year"].between(2020, 2024)) & (df_model["Community Area"] != 0)]
    .groupby("Community Area")
    .agg(Population_Mean_2020_2024=("Population", "mean"))
    .reset_index()
)

community_area_summary = population_area.merge(rates_area, on="Community Area", how="left")
community_area_summary["Population_Mean_2020_2024"] = (
    community_area_summary["Population_Mean_2020_2024"].round(0).astype(int)
)

community_area_summary.head()

,Community Area,Population_Mean_2020_2024,Total_Crimes,Arrest_Rate,Domestic_Rate,Violent_Rate
0,1,54024,20350,26.20,14.53,31.77
1,2,78390,17577,17.98,13.63,28.40
2,3,54491,20242,29.09,8.99,27.17
3,4,41565,10181,17.81,9.29,24.72
4,5,36047,7954,13.96,6.25,16.58


In [25]:
community_area_summary.to_csv(
    "../outputs/community_area_summary_2020_2024.csv",
    index=False, sep=";", decimal=","
)

## Conclusiones del modelo

- El **Random Forest** (R² = 0.977) supera ampliamente tanto a la Regresión Lineal (R² = 0.63)
  como a un modelo base (R² ≈ 0), confirmando que la relación entre variables socioeconómicas
  y criminalidad no es lineal y que el modelo aporta valor predictivo real.
- **Población**, **tasa de desempleo** y **tasa de pobreza** son las variables más relevantes.
- La proyección a 2025-2027 mantiene los totales de crímenes en un rango similar al de 2024,
  consistente con la tendencia bajista observada en el EDA.
- El área de **Austin** se mantiene consistentemente como la zona con mayor número de crímenes
  predichos en todo el periodo 2025-2027.
